# ARES: Fast Evaluation for Remaining Benchmark Domains (Qwen2.5-7B-Instruct 4-bit)

This dedicated notebook loads the pre-trained checkpoints (GRM, LRM, Router, 5 LoRA Experts) and evaluates the remaining benchmark domains (**GSM8K Math, MBPP Code, AI2-ARC Science, CommonsenseQA Reasoning**).

- Automatically resumes from `outputs/benchmark_checkpoint_7b_test_500.json` (skips the already completed 450 WikiText General samples).
- Uses unbuffered stdout (`!python -u`) for live real-time log streaming.
- Completes in ~1 hour on dual T4 GPU without any timeout risk.

In [ ]:
# === [1/3] Environment Setup & Sync Repository ===
!if [ -d "/kaggle/working/ARES-research" ]; then cd /kaggle/working/ARES-research && git pull origin main; else cd /kaggle/working && git clone https://github.com/sharksurfauto-byte/ARES-research.git; fi

%cd /kaggle/working/ARES-research
!pip install -q --upgrade pip
!pip install -q -e .

import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("=== Environment Ready ===")

In [ ]:
# === [2/3] Checkpoint Linking & Progress Resumption Setup ===
import os, shutil
from pathlib import Path

print("Checking for pre-trained checkpoints...")

# Search input paths for checkpoints if attached as Kaggle dataset or notebook
kaggle_inputs = list(Path("/kaggle/input").glob("*"))
print(f"Detected Kaggle input folders: {[p.name for p in kaggle_inputs]}")

for inp in kaggle_inputs:
    # 1. Look for checkpoints directory
    for candidate in [inp / "checkpoints", inp / "ARES-research" / "checkpoints"]:
        if candidate.exists() and (candidate / "reliability" / "grm.pt").exists():
            print(f"[Found Checkpoints] Copying from {candidate} to ./checkpoints/ ...")
            shutil.copytree(candidate, "checkpoints", dirs_exist_ok=True)
            break
            
    # 2. Look for existing benchmark checkpoint file
    for candidate_file in [
        inp / "benchmark_checkpoint_7b_test_500.json",
        inp / "outputs" / "benchmark_checkpoint_7b_test_500.json",
        inp / "ARES-research" / "outputs" / "benchmark_checkpoint_7b_test_500.json",
    ]:
        if candidate_file.exists():
            print(f"[Found Progress Checkpoint] Copying {candidate_file.name} to ./outputs/ ...")
            os.makedirs("outputs", exist_ok=True)
            shutil.copyfile(candidate_file, "outputs/benchmark_checkpoint_7b_test_500.json")
            break

# Verify critical checkpoints exist
assert Path("checkpoints/reliability/grm.pt").exists(), "ERROR: checkpoints/reliability/grm.pt not found! Please attach dataset or previous notebook."
assert Path("checkpoints/reliability/lrm.pt").exists(), "ERROR: checkpoints/reliability/lrm.pt not found!"
assert Path("checkpoints/router/router.pt").exists() or Path("checkpoints/router/router_best.pt").exists(), "ERROR: router checkpoint not found!"
print("\nSUCCESS: All pre-trained checkpoints (GRM, LRM, Router, 5 Experts) verified!")

In [ ]:
# === [3/3] Fast Benchmark Evaluation (GSM8K, MBPP, AI2-ARC, CommonsenseQA) ===
# Note: !python -u streams live stdout directly to the notebook log with zero buffering delay

!python -u scripts/run_ares_pipeline.py \
    --model_name "Qwen/Qwen2.5-7B-Instruct" \
    --grm_checkpoint "checkpoints/reliability/grm.pt" \
    --lrm_checkpoint "checkpoints/reliability/lrm.pt" \
    --router_checkpoint "checkpoints/router/router_best.pt" \
    --expert_dir "checkpoints/experts" \
    --benchmark all \
    --split test \
    --samples_per_domain 100 \
    --max_new_tokens 64 \
    --checkpoint_every 20 \
    --checkpoint_file "outputs/benchmark_checkpoint_7b_test_500.json" \
    --run_baselines \
    --output_report "benchmarks_ares_report.md" \
    --output_json "benchmarks_ares_results.json" \
    --device cuda

print("\n" + "=" * 70)
print("=== ARES BENCHMARK EVALUATION SUMMARY REPORT ===")
print("=" * 70)
!cat benchmarks_ares_report.md